We use log transformation on rent because the rental data is highly skewed, with a few very expensive properties. Log transformation reduces the effect of these extreme values and helps Linear Regression make more balanced predictions for typical rental properties.

🟢 Raw rent = easier, but affected more by extreme expensive rents

🟢 Log rent = compresses extreme values and can give more balanced predictions

🟢 For Rentora AI + Linear Regression, log rent is a good choice to try

## Model Training — Linear Regression Baseline

Training on `log_rent` (compresses the heavy right-skew in rent — see EDA notebook) 
using `rentora_eda_final.csv`. This documents the baseline attempt, the multicollinearity 
issue found along the way, and why Linear Regression underperforms on this data — 
setting up Random Forest / XGBoost next.

In [2]:
#imports and load
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

df = pd.read_csv('../data/rentora_eda_final.csv')
df.shape

(117375, 14)

In [3]:
#create target, split
df['log_rent'] = np.log1p(df['rent'])

base_cols = ['city', 'locality', 'bhk', 'size_sqft', 'furnishing', 'bathrooms',
             'latitude', 'longitude', 'near_highway', 'near_mall', 'near_river', 'near_mountain']

X = df[base_cols].copy()
y = df['log_rent']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=df['city']
)

X_train.shape, X_test.shape

((93900, 12), (23475, 12))

In [4]:
#target encode locality (train-only, with fallback for unseen localities)
# Target encoding: average rent per locality, computed from TRAINING data only (avoids leakage)
locality_avg_rent = df.loc[X_train.index].groupby('locality')['rent'].mean()

X_train['locality_encoded'] = X_train['locality'].map(locality_avg_rent)
X_test['locality_encoded'] = X_test['locality'].map(locality_avg_rent)

# Fallback for localities in test set never seen during training
overall_avg = df.loc[X_train.index]['rent'].mean()
X_test['locality_encoded'] = X_test['locality_encoded'].fillna(overall_avg)

X_train = X_train.drop(columns=['locality'])
X_test = X_test.drop(columns=['locality'])

print("Nulls in test after fillna:", X_test['locality_encoded'].isna().sum())

Nulls in test after fillna: 0


In [5]:
#one-hot encode city and furnishing
X_train = pd.get_dummies(X_train, columns=['city', 'furnishing'], drop_first=True)
X_test = pd.get_dummies(X_test, columns=['city', 'furnishing'], drop_first=True)
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

X_train.columns.tolist()

['bhk',
 'size_sqft',
 'bathrooms',
 'latitude',
 'longitude',
 'near_highway',
 'near_mall',
 'near_river',
 'near_mountain',
 'locality_encoded',
 'city_Bangalore',
 'city_Chennai',
 'city_Delhi',
 'city_Hyderabad',
 'city_Kolkata',
 'city_Mumbai',
 'city_Pune',
 'furnishing_Semi-Furnished',
 'furnishing_Unfurnished']

### Attempt 1: raw features, no scaling

In [6]:
#Linear Regression attempt 1
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_rent = np.expm1(lr.predict(X_test))
y_test_rent = np.expm1(y_test)

rmse = np.sqrt(mean_squared_error(y_test_rent, y_pred_rent))
mae = mean_absolute_error(y_test_rent, y_pred_rent)
r2 = r2_score(y_test_rent, y_pred_rent)

print(f"RMSE: ₹{rmse:,.2f} | MAE: ₹{mae:,.2f} | R²: {r2:.4f}")
print("Predictions > 1 crore:", (y_pred_rent > 1e7).sum())

RMSE: ₹2,213,275.57 | MAE: ₹52,383.57 | R²: -511.1997
Predictions > 1 crore: 19


Predictions blew up (up to ₹3 crore/month). Suspect: unscaled features with very 
different ranges (`locality_encoded` up to 11 lakh vs `bhk` 1–5) destabilizing the model.

### Attempt 2: scaling numeric features

In [7]:
#scale, retrain
scaler = StandardScaler()
numeric_cols = ['bhk', 'size_sqft', 'bathrooms', 'latitude', 'longitude', 'locality_encoded']

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

lr2 = LinearRegression()
lr2.fit(X_train_scaled, y_train)
y_pred_rent2 = np.expm1(lr2.predict(X_test_scaled))

rmse2 = np.sqrt(mean_squared_error(y_test_rent, y_pred_rent2))
mae2 = mean_absolute_error(y_test_rent, y_pred_rent2)
r22 = r2_score(y_test_rent, y_pred_rent2)

print(f"RMSE: ₹{rmse2:,.2f} | MAE: ₹{mae2:,.2f} | R²: {r22:.4f}")
print("Predictions > 1 crore:", (y_pred_rent2 > 1e7).sum())

RMSE: ₹2,104,321.82 | MAE: ₹50,259.94 | R²: -462.0124
Predictions > 1 crore: 18


Scaling alone didn't fix it — checking Variance Inflation Factor (VIF) for multicollinearity.

In [8]:
#VIF check
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_train_vif = X_train_scaled.astype(float)
vif_data = pd.DataFrame()
vif_data["feature"] = X_train_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_train_vif.values, i) for i in range(X_train_vif.shape[1])]
vif_data.sort_values("VIF", ascending=False)

,feature,VIF
3,latitude,2750.865446
12,city_Delhi,1946.193395
4,longitude,1104.105739
14,city_Kolkata,959.789674
15,city_Mumbai,467.311801
10,city_Bangalore,449.335346
16,city_Pune,191.205454
11,city_Chennai,86.586268
13,city_Hyderabad,5.793594
0,bhk,5.271800


VIF confirmed severe multicollinearity: `latitude` (≈2,750), `city_Delhi` (≈1,946), 
`longitude` (≈1,104) — all encoding the same location signal redundantly. 
`locality_encoded` scored ≈2.2 (healthy, genuinely independent) — keeping it as the 
sole location feature.

### Attempt 3: drop redundant location features

In [9]:
#drop redundant columns, retrain
drop_cols = ['latitude', 'longitude', 'city_Bangalore', 'city_Chennai', 'city_Delhi',
             'city_Hyderabad', 'city_Kolkata', 'city_Mumbai', 'city_Pune']

X_train_v2 = X_train_scaled.drop(columns=drop_cols)
X_test_v2 = X_test_scaled.drop(columns=drop_cols)

lr3 = LinearRegression()
lr3.fit(X_train_v2, y_train)
y_pred_rent3 = np.expm1(lr3.predict(X_test_v2))

rmse3 = np.sqrt(mean_squared_error(y_test_rent, y_pred_rent3))
mae3 = mean_absolute_error(y_test_rent, y_pred_rent3)
r23 = r2_score(y_test_rent, y_pred_rent3)

print(f"RMSE: ₹{rmse3:,.2f} | MAE: ₹{mae3:,.2f} | R²: {r23:.4f}")
print("Predictions > 1 crore:", (y_pred_rent3 > 1e7).sum())

RMSE: ₹1,436,364.33 | MAE: ₹49,765.40 | R²: -214.7235
Predictions > 1 crore: 19


### Still broken — further diagnosis
Ruled out: locality sample-count overfitting (extreme predictions map to well-sampled 
localities too). Checked: model coefficients (all reasonable, no blowup). Checked: 
raw values behind the extreme predictions directly.

In [10]:
#coefficient check + raw value inspection (combined)
coef_df = pd.DataFrame({'feature': X_train_v2.columns, 'coefficient': lr3.coef_})
coef_df['abs_coef'] = coef_df['coefficient'].abs()
print(coef_df.sort_values('abs_coef', ascending=False))

extreme_idx = X_test_v2[y_pred_rent3 > 1e7].index
raw_extreme = df.loc[extreme_idx, ['city', 'locality', 'bhk', 'size_sqft', 'bathrooms', 'rent']]
print("\nRaw values behind extreme predictions:")
print(raw_extreme.sort_values('bhk', ascending=False))

                     feature  coefficient  abs_coef
7           locality_encoded     0.422980  0.422980
9     furnishing_Unfurnished    -0.386109  0.386109
6              near_mountain     0.373111  0.373111
8  furnishing_Semi-Furnished    -0.270309  0.270309
2                  bathrooms     0.218283  0.218283
5                 near_river    -0.182287  0.182287
1                  size_sqft     0.181520  0.181520
0                        bhk     0.180641  0.180641
4                  near_mall    -0.101922  0.101922
3               near_highway     0.055137  0.055137

Raw values behind extreme predictions:
             city            locality   bhk  size_sqft  bathrooms       rent
57244     Kolkata            New Town  15.0    10000.0       16.0   350000.0
18076   Bangalore            Nagawara  15.0     6500.0       19.0   200000.0
40547       Delhi  Panchsheel Enclave  15.0    12921.0       15.0   818000.0
36814       Delhi  New Friends Colony  15.0    15461.0       16.0  1118000.0
405

Confirmed: the extreme predictions are genuine ultra-luxury Delhi properties 
(Panchsheel Enclave, Prithviraj Road — real government-bungalow-tier addresses), 
not data errors.

### Final check: is this just the luxury tail, or a broader problem?

In [11]:
#residual/ratio analysis
mask = y_pred_rent3 <= 1e7
residuals = y_test_rent[mask] - y_pred_rent3[mask]
ratio = y_pred_rent3[mask] / y_test_rent[mask]

r2_clean = r2_score(y_test_rent[mask], y_pred_rent3[mask])
print(f"R² excluding {(~mask).sum()} extreme predictions: {r2_clean:.4f}")
print("Predictions <50% or >200% of actual rent:", ((ratio < 0.5) | (ratio > 2)).sum(), "out of", mask.sum())

R² excluding 19 extreme predictions: -0.8022
Predictions <50% or >200% of actual rent: 3317 out of 23456


## Conclusion: Linear Regression baseline

Even excluding extreme luxury cases, R² remains negative and **14% of predictions 
are off by more than 2x or under 0.5x actual rent**. Root cause: rent depends on 
multiplicative interactions (locality prestige × size) that a linear model structurally 
cannot represent. This is the baseline to beat — moving to Random Forest next.

In [12]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=100,       # half as many trees
    max_depth=18,           # caps tree depth — biggest single lever for file size
    min_samples_leaf=3,     # prevents tiny, overly-specific leaf nodes
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train_v2, y_train)

y_pred_rent_rf = np.expm1(rf.predict(X_test_v2))

rmse_rf = np.sqrt(mean_squared_error(y_test_rent, y_pred_rent_rf))
mae_rf = mean_absolute_error(y_test_rent, y_pred_rent_rf)
r2_rf = r2_score(y_test_rent, y_pred_rent_rf)

print(f"RMSE: ₹{rmse_rf:,.2f} | MAE: ₹{mae_rf:,.2f} | R²: {r2_rf:.4f}")

RMSE: ₹38,085.85 | MAE: ₹12,329.68 | R²: 0.8483


This validates the whole "multiplicative interactions" theory — Random Forest can naturally represent "locality prestige × size" type relationships by splitting on one feature within branches already split by another, which Linear Regression simply cannot do.

In [13]:
#A few things worth doing before moving to XGBoost:
#1. Check for extreme predictions again — did the crore-plus blowups disappear entirely, or just shrink?
print("Predictions > 1 crore:", (y_pred_rent_rf > 1e7).sum())
print("Predicted rent range:", y_pred_rent_rf.min(), "to", y_pred_rent_rf.max())

Predictions > 1 crore: 0
Predicted rent range: 3303.149787650398 to 1199493.5432266253


2. Feature importance — this is genuinely useful for your project beyond just model performance; it tells you which inputs actually matter for your chatbot's predictions, and whether the geo-features (near_highway, near_mall, etc.) you spent so much effort calibrating are pulling their weight:

In [14]:
importance_df = pd.DataFrame({
    'feature': X_train_v2.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)
print(importance_df)

                     feature  importance
7           locality_encoded    0.604858
1                  size_sqft    0.232938
2                  bathrooms    0.102426
0                        bhk    0.038114
8  furnishing_Semi-Furnished    0.005282
9     furnishing_Unfurnished    0.004903
6              near_mountain    0.004669
3               near_highway    0.003873
5                 near_river    0.002443
4                  near_mall    0.000496


## Random Forest results

RMSE: ₹38,862.74 | MAE: ₹12,603.33 | R²: 0.8421

Massive improvement over Linear Regression (R² went from negative to 0.84). Confirms 
the hypothesis: rent depends on multiplicative feature interactions (locality × size, 
etc.) that tree-based models capture naturally but linear models cannot. Moving to 
XGBoost next to see if boosting improves further.

In [15]:
import xgboost as xgb
# Try a lower learning rate with more trees — often improves XGBoost meaningfully
xgb_model3 = xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=10, random_state=42, n_jobs=-1)
xgb_model3.fit(X_train_v2, y_train)

y_pred_rent_xgb3 = np.expm1(xgb_model3.predict(X_test_v2))
rmse3 = np.sqrt(mean_squared_error(y_test_rent, y_pred_rent_xgb3))
mae3 = mean_absolute_error(y_test_rent, y_pred_rent_xgb3)
r23 = r2_score(y_test_rent, y_pred_rent_xgb3)

print(f"RMSE: ₹{rmse3:,.2f} | MAE: ₹{mae3:,.2f} | R²: {r23:.4f}")

RMSE: ₹39,201.55 | MAE: ₹12,462.11 | R²: 0.8393


## Final model comparison

| Model | RMSE | MAE | R² |
|---|---|---|---|
| Linear Regression | ₹1,26,162+ (14% predictions off 2x+) | — | negative |
| **Random Forest** | **₹38,862.74** | **₹12,603.33** | **0.8421** |
| XGBoost (3 configs tried) | ₹39,089–39,253 | ₹12,462–12,819 | 0.8389–0.8402 |

**Random Forest is the winning model** — R² 0.8421, MAE ₹12,603, outperforming both 
Linear Regression (structurally unable to capture multiplicative locality×size 
interactions) and three tuned XGBoost configurations. Model saved for use in the 
FastAPI /predict endpoint.

In [16]:
import os
os.makedirs('../backend', exist_ok=True)

In [17]:
import joblib

joblib.dump(rf, '../backend/model.pkl')
joblib.dump(scaler, '../backend/scaler.pkl')
joblib.dump(locality_avg_rent, '../backend/locality_avg_rent.pkl')
joblib.dump(overall_avg, '../backend/overall_avg.pkl')
joblib.dump(list(X_train_v2.columns), '../backend/model_columns.pkl')

print("Saved model and preprocessing artifacts")

Saved model and preprocessing artifacts


In [19]:
from sklearn.ensemble import RandomForestRegressor
import joblib
import os

model = RandomForestRegressor(
    n_estimators=100,
    max_depth=18,
    min_samples_leaf=3,
    random_state=42
)
model.fit(X_train, y_train)  # use whatever your actual training vars are named

joblib.dump(model, 'model_lite_test.pkl')
size_mb = os.path.getsize('model_lite_test.pkl') / (1024 * 1024)
print(f"New model size: {size_mb:.2f} MB")

New model size: 144.37 MB


In [20]:
import os
size_mb = os.path.getsize('../backend/model.pkl') / (1024 * 1024)
print(f"backend/model.pkl size: {size_mb:.2f} MB")

backend/model.pkl size: 138.95 MB
